In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("spotify dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully!
Shape: (32833, 23)


,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


In [3]:
print("Columns in the dataset:")
print(df.columns.tolist())

Columns in the dataset:
['track_id', 'track_name', 'track_artist', 'track_popularity', 'track_album_id', 'track_album_name', 'track_album_release_date', 'playlist_name', 'playlist_id', 'playlist_genre', 'playlist_subgenre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']


In [4]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

# Keep only features that exist in the dataset
available_features = [feature for feature in audio_features if feature in df.columns]

print("Features used for recommendation:")
print(available_features)

Features used for recommendation:
['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


In [5]:
recommendation_data = df[available_features].copy()

# Convert values to numeric
recommendation_data = recommendation_data.apply(pd.to_numeric, errors="coerce")

# Remove rows with missing feature values
valid_rows = recommendation_data.dropna().index

recommendation_data = recommendation_data.loc[valid_rows]
songs = df.loc[valid_rows].copy()

print("Songs available for recommendation:", len(songs))
print("Features used:", available_features)

Songs available for recommendation: 32833
Features used: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


In [6]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(recommendation_data)

print("Audio features scaled successfully!")
print("Scaled data shape:", scaled_features.shape)

Audio features scaled successfully!
Scaled data shape: (32833, 9)


In [7]:
similarity_matrix = cosine_similarity(scaled_features)

print("Similarity matrix created successfully!")
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix created successfully!
Similarity matrix shape: (32833, 32833)


In [8]:
def recommend_songs(song_name, number_of_recommendations=10):
    matches = songs[
        songs["track_name"].astype(str).str.lower() == song_name.lower()
    ]

    if matches.empty:
        print(f"Song '{song_name}' was not found in the dataset.")
        return pd.DataFrame()

    song_index = matches.index[0]
    position = songs.index.get_loc(song_index)

    similarity_scores = similarity_matrix[position]

    similar_positions = np.argsort(similarity_scores)[::-1]

    # Remove the selected song itself
    similar_positions = [
        i for i in similar_positions if i != position
    ]

    selected_positions = similar_positions[:number_of_recommendations]

    recommendations = songs.iloc[selected_positions].copy()
    recommendations["similarity_score"] = [
        similarity_scores[i] for i in selected_positions
    ]

    return recommendations

In [9]:
songs[["track_name", "track_artist"]].head(10)

,track_name,track_artist
0,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran
1,Memories - Dillon Francis Remix,Maroon 5
2,All the Time - Don Diablo Remix,Zara Larsson
3,Call You Mine - Keanu Silva Remix,The Chainsmokers
4,Someone You Loved - Future Humans Remix,Lewis Capaldi
5,Beautiful People (feat. Khalid) - Jack Wins Remix,Ed Sheeran
6,Never Really Over - R3HAB Remix,Katy Perry
7,Post Malone (feat. RANI) - GATTÜSO Remix,Sam Feldt
8,Tough Love - Tiësto Remix / Radio Edit,Avicii
9,If I Can't Have You - Gryffin Remix,Shawn Mendes


In [10]:
recommend_songs("PUT SONG NAME HERE", 10)

Song 'PUT SONG NAME HERE' was not found in the dataset.


""


In [11]:
songs[["track_name", "track_artist"]].head(20)

,track_name,track_artist
0,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran
1,Memories - Dillon Francis Remix,Maroon 5
2,All the Time - Don Diablo Remix,Zara Larsson
3,Call You Mine - Keanu Silva Remix,The Chainsmokers
4,Someone You Loved - Future Humans Remix,Lewis Capaldi
5,Beautiful People (feat. Khalid) - Jack Wins Remix,Ed Sheeran
6,Never Really Over - R3HAB Remix,Katy Perry
7,Post Malone (feat. RANI) - GATTÜSO Remix,Sam Feldt
8,Tough Love - Tiësto Remix / Radio Edit,Avicii
9,If I Can't Have You - Gryffin Remix,Shawn Mendes


In [12]:
recommend_songs("Hate Me - R3HAB Remix", 10)

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,similarity_score
29678,3iJeF2izF1QcpCgUc97BLl,Hate Me - R3HAB Remix,Ellie Goulding,67,6e6RVFWdZaoTiLAZOCMnps,Hate Me (R3HAB Remix),2019-08-16,Pop EDM Remixes,4aUEH3uhbofktrFkXOOaKj,edm,...,-4.918,1,0.0554,0.148000,0.000000,0.1330,0.329,123.125,203733,1.000000
32180,0B8tteaC3h0B5EkeA4BAWN,Lost In Your Love - Alkaz Remix,Friendzone,42,6ni1QcWu53oxMATfXTw25t,Lost In Your Love (Alkaz Remix),2019-12-27,Brand New EDM,2dNitDEHM9FpUGEHWc7zyW,edm,...,-4.225,0,0.0460,0.137000,0.000000,0.1030,0.371,127.983,200625,0.974847
31342,5ETRYg14nPQtgoNrjvW4eY,Believe,Kosling,38,1yxshZc0H3QiRw4hUOf3H8,Believe,2018-04-06,Female Vocal EDM,5QaPRbVgbEcCc9jC1UxgD1,edm,...,-3.929,1,0.0449,0.170000,0.014400,0.0798,0.331,127.990,200861,0.973396
31504,0CJzVKPtJaeOt76jO3PpbZ,I Lived - Arty Remix,OneRepublic,0,41GUlM6LA1iq2piEp9G9bx,I Lived (Arty Remix),2014-09-23,Vocal House,5PCAWKfUWAUj8VeY8G7xRQ,edm,...,-4.070,1,0.0530,0.231000,0.000000,0.0790,0.106,127.931,267427,0.973214
30898,4TNWgANrFMzz9qfPlBNopm,Satellite,Tritonal,35,0OfEsPc1vhiRsuTN2bCERz,Metamorphic III,2014-09-29,Happy EDM,3d2JFEbvcx7p2CHkBaQeMZ,edm,...,-3.745,0,0.0502,0.043600,0.000005,0.0756,0.207,128.091,241875,0.970099
15079,0c3fJd2HRi6uHWgTljIae5,Dear Agony,Breaking Benjamin,60,0mBvlBWI6TMDrWvQ8bKOKV,Dear Agony,2009-01-01,Hard Rock Workout,3YouF0u7waJnolytf9JCXf,rock,...,-3.572,1,0.0414,0.000085,0.000000,0.1130,0.218,125.103,258253,0.964870
21731,5rgnyXr2PXFWsAnPFew077,I Was Never There,The Weeknd,17,3N88bRVAwQrtKqSV0UgU69,"My Dear Melancholy,",2018-03-30,Urban Contemporary,4Pbs84EQbuAblxlp6Chz0d,r&b,...,-4.054,1,0.0337,0.135000,0.000048,0.1690,0.169,113.923,241067,0.964844
28362,1ZF65jRYkGZhnfrRzjug4g,Black Butterflies,Kill The Buzz,38,0ZBmMXprkNO8WZNuFF7Umy,Black Butterflies,2019-05-29,Big Room EDM,3S03NWbWkrR1e7DPH9rf9Z,edm,...,-4.762,1,0.0520,0.030300,0.000000,0.1260,0.300,127.770,204375,0.960802
31322,55DWZ04d9zY2N7Nr9xyzt0,Light Me Up,Matisse & Sadko,0,4W8lIWDfuhnFPCa1iI1FvH,Light Me Up,2018-08-03,Female Vocal EDM,5QaPRbVgbEcCc9jC1UxgD1,edm,...,-5.128,1,0.0432,0.038600,0.000000,0.1120,0.299,126.107,208333,0.960552
32011,2PLRrQ8oASLJIxw2mwOvh6,Love Is on Our Side,Capital Kings,0,0yYhkVzDvpHYckwcrAOFJO,Love Is on Our Side,2017-06-02,CHRISTIAN ELECTRO / DANCE / EDM,0MhTMIo1bgH6zzPh7BdChT,edm,...,-4.466,0,0.0439,0.167000,0.000000,0.0828,0.332,134.949,199309,0.959505


# Recommendation System — Part 4

## Objective

The recommendation system suggests songs that are acoustically similar to a selected song.

## Methodology

The system uses Spotify audio features such as:

- Danceability
- Energy
- Loudness
- Speechiness
- Acousticness
- Instrumentalness
- Liveness
- Valence
- Tempo

The features are standardized using `StandardScaler`.

Cosine similarity is then used to measure the similarity between songs.

## Recommendation Process

Selected Song
↓
Audio Feature Extraction
↓
Feature Scaling
↓
Cosine Similarity
↓
Similarity Ranking
↓
Top 10 Recommended Songs

## Output

The system returns the most similar songs along with their artist, playlist genre, and similarity score.

## Role in the Group Project

This module represents **Part 4 — Recommendation System Builder** and can later be integrated into the Streamlit website developed by the team.